# CSCI 3151 â€” Foundations of Machine Learning

### M34 â€” Vanishing & Exploding Gradients: Problems and Remedies

In this notebook you will **experiment with deep ReLU networks** and see how
initialization, normalization, and residual connections affect gradient flow
and training stability.


## Assumptions & scope

You should already be comfortable with:

- Basic Python, NumPy, and Matplotlib
- Training simple models in PyTorch (logistic regression / small MLPs)
- The idea of backpropagation and gradient norms (from M31â€“M33)

The focus here is on **coding experiments** rather than long written answers.


In [ ]:
# Setup: imports and global configuration

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.datasets import make_moons

# Reproducibility
torch.manual_seed(3151)
np.random.seed(3151)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

plt.style.use("default")


In [ ]:
# Generate a simple non-linear 2D dataset (two moons)

X_np, y_np = make_moons(n_samples=2000, noise=0.2, random_state=3151)

X = torch.from_numpy(X_np).float().to(device)
y = torch.from_numpy(y_np).long().to(device)

print("X shape:", X.shape)
print("y shape:", y.shape)

# Quick visualization (on CPU)
plt.figure(figsize=(4, 4))
plt.scatter(X_np[:, 0], X_np[:, 1], c=y_np, s=10, cmap="coolwarm", alpha=0.6)
plt.title("Two moons dataset")
plt.xlabel("x1")
plt.ylabel("x2")
plt.tight_layout()
plt.show()


In [ ]:
# Define deep MLP architectures and utilities for initialization & gradient stats

class DeepMLP(nn.Module):
    def __init__(
        self,
        in_dim,
        hidden_dim,
        out_dim,
        depth=30,
        activation="relu",
        init_scheme="he",
        use_batchnorm=False,
        use_residual=False,
    ):
        super().__init__()
        self.depth = depth
        self.activation_name = activation
        self.use_batchnorm = use_batchnorm
        self.use_residual = use_residual

        layers = []
        bns = []

        dims = [in_dim] + [hidden_dim] * (depth - 1) + [out_dim]
        for i in range(len(dims) - 1):
            layers.append(nn.Linear(dims[i], dims[i + 1]))
            # BatchNorm only on hidden layers
            if use_batchnorm and i < depth - 1:
                bns.append(nn.BatchNorm1d(dims[i + 1]))

        self.layers = nn.ModuleList(layers)
        self.bns = nn.ModuleList(bns)

        self._init_weights(init_scheme)

    def _activation(self, x):
        if self.activation_name.lower() == "relu":
            return F.relu(x)
        elif self.activation_name.lower() == "tanh":
            return torch.tanh(x)
        else:
            return x  # identity as a fallback

    def _init_weights(self, scheme):
        """
        Initialize all linear layers.

        TODO:
        - Make sure you can explain the difference between the schemes below.
        - Try adding your own scheme (e.g., 'very_small' or 'orthogonal').
        """
        for layer in self.layers:
            if scheme == "small":
                nn.init.normal_(layer.weight, mean=0.0, std=0.01)
            elif scheme == "large":
                nn.init.normal_(layer.weight, mean=0.0, std=1.0)
            elif scheme == "xavier":
                nn.init.xavier_uniform_(layer.weight)
            elif scheme == "he":
                nn.init.kaiming_uniform_(layer.weight, nonlinearity="relu")
            else:
                # Default to something reasonable
                nn.init.kaiming_uniform_(layer.weight, nonlinearity="relu")

            nn.init.zeros_(layer.bias)

    def forward(self, x):
        h = x
        bn_idx = 0
        for i, layer in enumerate(self.layers):
            h_in = h
            h = layer(h)

            is_output_layer = i == len(self.layers) - 1
            if not is_output_layer:
                if self.use_batchnorm:
                    h = self.bns[bn_idx](h)
                    bn_idx += 1
                h = self._activation(h)
                if self.use_residual and h.shape == h_in.shape:
                    h = h + h_in
        return h


def one_training_step(model, X, y, optimizer, loss_fn):
    """
    Run a single training step and return the loss and gradient norms
    of the *first* and *last* layer.

    NOTE: we'll reuse this for multiple experiments.
    """
    model.train()
    optimizer.zero_grad()

    logits = model(X)
    loss = loss_fn(logits, y)
    loss.backward()

    with torch.no_grad():
        first_grad = model.layers[0].weight.grad.norm().item()
        last_grad = model.layers[-1].weight.grad.norm().item()

    optimizer.step()

    return loss.item(), first_grad, last_grad


## Core Task 1 â€” Initialization vs training stability

In lecture we argued that **initialization** strongly affects whether gradients
vanish, explode, or stay in a healthy range.

In this experiment you will train a deep ReLU network with several
initialization schemes and compare:

- training loss curves
- gradient norms at the first and last layer


In [ ]:
# Experiment: compare initialization schemes for a deep ReLU network

depth = 30        # feel free to change
hidden_dim = 64
lr = 0.1
n_steps = 200

init_schemes = ["small", "xavier", "he", "large"]  # TODO: add/remove schemes

results = {}

loss_fn = nn.CrossEntropyLoss()

for scheme in init_schemes:
    print(f"\n=== init_scheme = {scheme} ===")
    model = DeepMLP(
        in_dim=2,
        hidden_dim=hidden_dim,
        out_dim=2,
        depth=depth,
        activation="relu",
        init_scheme=scheme,
        use_batchnorm=False,
        use_residual=False,
    ).to(device)

    optimizer = torch.optim.SGD(model.parameters(), lr=lr)

    losses = []
    first_grads = []
    last_grads = []

    for step in range(n_steps):
        loss, g_first, g_last = one_training_step(model, X, y, optimizer, loss_fn)
        losses.append(loss)
        first_grads.append(g_first)
        last_grads.append(g_last)

    results[scheme] = {
        "loss": np.array(losses),
        "grad_first": np.array(first_grads),
        "grad_last": np.array(last_grads),
    }

# Quick plots: loss and gradient norms
steps = np.arange(1, n_steps + 1)

plt.figure(figsize=(12, 4))

plt.subplot(1, 3, 1)
for scheme in init_schemes:
    plt.plot(steps, results[scheme]["loss"], label=scheme)
plt.xlabel("Step")
plt.ylabel("Loss")
plt.title("Training loss")
plt.yscale("log")
plt.grid(True, ls=":")
plt.legend()

plt.subplot(1, 3, 2)
for scheme in init_schemes:
    plt.plot(steps, results[scheme]["grad_first"], label=scheme)
plt.xlabel("Step")
plt.ylabel("||âˆ‚L/âˆ‚W_first||")
plt.title("Gradient norm â€” first layer")
plt.yscale("log")
plt.grid(True, ls=":")

plt.subplot(1, 3, 3)
for scheme in init_schemes:
    plt.plot(steps, results[scheme]["grad_last"], label=scheme)
plt.xlabel("Step")
plt.ylabel("||âˆ‚L/âˆ‚W_last||")
plt.title("Gradient norm â€” last layer")
plt.yscale("log")
plt.grid(True, ls=":")

plt.tight_layout()
plt.show()

# TODO:
#   Change `depth`, `hidden_dim`, `lr`, or `n_steps`, re-run,
#   and see which schemes are stable or unstable.


ðŸ“ **Core Task 1 â€” Your observations**

Use this space (in your own copy of the notebook) to jot down what
you see for different initialization schemes:

- When do gradients vanish or explode?
- Which schemes keep gradients in a healthy range?
- How does changing depth / width affect things?


## Core Task 2 â€” Batch normalization as a remedy

Now we compare a plain deep ReLU network to the same architecture with
**BatchNorm** after each hidden layer.

Your goal is to see how BatchNorm changes:

- the speed of optimization
- the scale and stability of gradient norms


In [ ]:
# Experiment: plain vs BatchNorm for a deep ReLU network

depth = 30
hidden_dim = 64
lr = 0.1
n_steps = 200
init_scheme = "he"

configs = {
    "plain": dict(use_batchnorm=False),
    "batchnorm": dict(use_batchnorm=True),
}

loss_fn = nn.CrossEntropyLoss()
bn_results = {}

for name, cfg in configs.items():
    print(f"\n=== config = {name} ===")
    model = DeepMLP(
        in_dim=2,
        hidden_dim=hidden_dim,
        out_dim=2,
        depth=depth,
        activation="relu",
        init_scheme=init_scheme,
        use_batchnorm=cfg["use_batchnorm"],
        use_residual=False,
    ).to(device)

    optimizer = torch.optim.SGD(model.parameters(), lr=lr)

    losses = []
    first_grads = []
    last_grads = []

    for step in range(n_steps):
        loss, g_first, g_last = one_training_step(model, X, y, optimizer, loss_fn)
        losses.append(loss)
        first_grads.append(g_first)
        last_grads.append(g_last)

    bn_results[name] = {
        "loss": np.array(losses),
        "grad_first": np.array(first_grads),
        "grad_last": np.array(last_grads),
    }

steps = np.arange(1, n_steps + 1)

plt.figure(figsize=(12, 4))

plt.subplot(1, 3, 1)
for name in configs:
    plt.plot(steps, bn_results[name]["loss"], label=name)
plt.xlabel("Step")
plt.ylabel("Loss")
plt.title("Training loss")
plt.yscale("log")
plt.grid(True, ls=":")
plt.legend()

plt.subplot(1, 3, 2)
for name in configs:
    plt.plot(steps, bn_results[name]["grad_first"], label=name)
plt.xlabel("Step")
plt.ylabel("||âˆ‚L/âˆ‚W_first||")
plt.title("Gradient norm â€” first layer")
plt.yscale("log")
plt.grid(True, ls=":")

plt.subplot(1, 3, 3)
for name in configs:
    plt.plot(steps, bn_results[name]["grad_last"], label=name)
plt.xlabel("Step")
plt.ylabel("||âˆ‚L/âˆ‚W_last||")
plt.title("Gradient norm â€” last layer")
plt.yscale("log")
plt.grid(True, ls=":")

plt.tight_layout()
plt.show()

# TODO:
#   Try different depths and learning rates.
#   Does BatchNorm always help with stability?


ðŸ“ **Core Task 2 â€” Your observations**

Again, note what you see:

- Does BatchNorm change how quickly the loss drops?
- Does it keep gradient norms in a narrower, more stable range?
- Are there settings (depth / lr) where it helps a lot or not at all?


## Stretch Task â€” Residual connections

Residual (skip) connections, like those in ResNets, are another way to
help gradients flow through very deep networks.

In the code cell below, we reuse `DeepMLP` with `use_residual=True`
and compare to a plain network.


In [ ]:
# Example template: residual vs non-residual (feel free to modify)

depth = 40
hidden_dim = 64
lr = 0.1
n_steps = 200
init_scheme = "he"

configs = {
    "plain": dict(use_batchnorm=False, use_residual=False),
    "residual": dict(use_batchnorm=False, use_residual=True),
}

loss_fn = nn.CrossEntropyLoss()
res_results = {}

for name, cfg in configs.items():
    print(f"\n=== config = {name} ===")
    model = DeepMLP(
        in_dim=2,
        hidden_dim=hidden_dim,
        out_dim=2,
        depth=depth,
        activation="relu",
        init_scheme=init_scheme,
        use_batchnorm=cfg["use_batchnorm"],
        use_residual=cfg["use_residual"],
    ).to(device)

    optimizer = torch.optim.SGD(model.parameters(), lr=lr)

    losses = []
    first_grads = []
    last_grads = []

    for step in range(n_steps):
        loss, g_first, g_last = one_training_step(model, X, y, optimizer, loss_fn)
        losses.append(loss)
        first_grads.append(g_first)
        last_grads.append(g_last)

    res_results[name] = {
        "loss": np.array(losses),
        "grad_first": np.array(first_grads),
        "grad_last": np.array(last_grads),
    }

steps = np.arange(1, n_steps + 1)

plt.figure(figsize=(12, 4))

plt.subplot(1, 3, 1)
for name in configs:
    plt.plot(steps, res_results[name]["loss"], label=name)
plt.xlabel("Step")
plt.ylabel("Loss")
plt.title("Training loss")
plt.yscale("log")
plt.grid(True, ls=":")
plt.legend()

plt.subplot(1, 3, 2)
for name in configs:
    plt.plot(steps, res_results[name]["grad_first"], label=name)
plt.xlabel("Step")
plt.ylabel("||âˆ‚L/âˆ‚W_first||")
plt.title("Gradient norm â€” first layer")
plt.yscale("log")
plt.grid(True, ls=":")

plt.subplot(1, 3, 3)
for name in configs:
    plt.plot(steps, res_results[name]["grad_last"], label=name)
plt.xlabel("Step")
plt.ylabel("||âˆ‚L/âˆ‚W_last||")
plt.title("Gradient norm â€” last layer")
plt.yscale("log")
plt.grid(True, ls=":")

plt.tight_layout()
plt.show()

# TODO:
#   Try even deeper networks (e.g., depth=80) and see when residual
#   connections start to really matter.


## Wrap-up â€” Connecting back to the big picture

Take a moment to connect your experiments to the conceptual story:

- Initialization controls the *starting scale* of activations/gradients.
- Normalization layers reshape distributions during training.
- Residual connections create short gradient paths through deep nets.

These are all different ways of **engineering good gradient flow**.
